# Loading FHIR data into a FHIR server

Your project may involve ingesting data into FHIR format and sending it to a FHIR server. This notebook is a quick example of how this can be achieved with IRIS health. 

This tutorial will assume you have a FHIR server set up with IRIS-health as described in the [FHIR Server quickstart tutorial](../1-setup/1.1-setup-iris-and-fhir.md). If you have set up the FHIR server by different methods, the differences will be where the requests are sent (Server URL, username and password) but the content will be the same. If you need to create the FHIR server yourself (for example on a new InterSystems IRIS for Health deployment), see [Setting up a FHIR server](3.0-setup-fhir-server.ipynb). The Bulk Load section at the end is optional.

If you need to bulk load FHIR data, for example you have a >1GB file and wish to avoid latency or issues with POST requests, skip to the [Bulk Load](#bulk-load) section.  

## Using HTTP Requests

You can add a bundle of FHIR resources to a FHIR Server using HTTP requests - the FHIR server will add the resources to the correct place. For an example, we are going to add 5 patient bundles (plus hospital and practitioner bundles). These bundles provide the complete medical history of the patient. 

All the FHIR data used in this project has been synthetically generated by Synthea. The instructions for this are a [separate tutorial](../6-extras/create-synthetic-fhir-data.md), but are super simple - use the command: 

    docker run --rm -v $PWD/output:/output --name synthea-docker intersystemsdc/irisdemo-base-synthea:version-1.3.4 -p 5

Where -p denotes the number of patients to generate and `$PWD/output` is the output directory of the files (a new folder called output in your current working directory).


This has already been done for you, so there are 5 example patient resources in ../data/fhir. Let's work with these.

In [1]:
import os
fhir_data_path = "../data/fhir/"
os.listdir(fhir_data_path)

['hospitalInformation1760013140357.json',
 'Jarred626_Walsh511_ea6f7f29-8553-4456-8fa9-3d629ffcd5f7.json',
 'Krista314_Mosciski958_79fdc511-4556-435d-b922-7a55d0d6e409.json',
 'Kristofer887_Schowalter414_a7b87618-d13a-4f02-8974-010aa35fb759.json',
 'Marshall526_Mosciski958_dbc8773a-10ac-42e5-9c8c-c8d269ed9179.json',
 'Milissa240_Gaylord332_db7fc62f-0f83-484f-bf84-40e85fc26c4d.json',
 'practitionerInformation1760013140357.json']

In [2]:
import json
import requests
from requests.auth import HTTPBasicAuth

server_url = "http://localhost:52773/fhir/r4/"
username = "SuperUser"
password = "SYS"

# Iterate over the files
for file in os.listdir(fhir_data_path):
    # Read the File
    with open(fhir_data_path+file, "r")as f:
        json_data = json.load(f)
        
    res = requests.post( server_url,
                         json=json_data,
                         headers={"Content-Type": "application/fhir+json"}, 
                         auth=HTTPBasicAuth(username, password))
    print(res)
    if res.status_code!=200:
        print(res.json())

<Response [200]>


<Response [200]>


<Response [200]>


<Response [200]>


<Response [200]>


<Response [200]>
<Response [200]>


Lets just check the patient resources were added: 

In [3]:
endpoint = "Patient"

res = requests.get(server_url+endpoint, headers={"Accept": "application/fhir+json"}, auth=HTTPBasicAuth(username, password))
print(res)
#print(res.json())
data = res.json() 

for entry in data.get("entry", [])[-5:]: ## Get the last 5 entries
    patient = entry.get('resource', {}) ## Get the patient resource
    print(patient.get('id'), patient.get('name'), patient.get('gender')) ## Print the patient info


<Response [200]>
13 [{'use': 'official', 'family': 'Walsh511', 'given': ['Jarred626']}] male
279 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Krista314'], 'prefix': ['Mrs.']}, {'use': 'maiden', 'family': 'Frami345', 'given': ['Krista314'], 'prefix': ['Mrs.']}] female
885 [{'use': 'official', 'family': 'Schowalter414', 'given': ['Kristofer887'], 'prefix': ['Mr.']}] male
1068 [{'use': 'official', 'family': 'Mosciski958', 'given': ['Marshall526'], 'prefix': ['Mr.']}] male
1741 [{'use': 'official', 'family': 'Gaylord332', 'given': ['Milissa240'], 'prefix': ['Ms.']}] female


And there we have it! How to add FHIR resources into a FHIR server running with InterSystems IRIS for Health. 

Next, you can [query this data with Python](3.2-querying-fhir-data-with-python.ipynb) or [create your own FHIR resources](3.3-create-fhir-resources.ipynb).

# Bulk Load

To bulk load large datasets into the FHIR Server, you can use the built in from InterSystems IRIS for Health. The downside is that the files need to be on the same filesystem as InterSystems IRIS for Health, so you need to copy files into the docker container or deployed server. This section is optional, and the commands below are shown rather than run.

To copy the files into the InterSystems IRIS container/server, use `docker cp` (local) or `scp` (SSH access):

```bash
docker cp fhirdata iris-fhir:/tmp/fhirdata
```

```bash
scp -r fhirdata irisowner@123-45-67-890:/tmp/fhirdata
```


Then load the data with the IRIS class `HS.FHIRServer.Tools.DataLoader.SubmitResourceFiles()` (via the Python Native API): 

```python
import time 
import iris 
st = time.time()

conn_args = {"hostname": "localhost", "port":1972, "namespace":"FHIRSERVER", "username": "SuperUser", "password":"SYS"}

conn  =iris.connect(**conn_args)

irispy = iris.createIRIS(conn) 

status = irispy.classMethodString("HS.FHIRServer.Tools.DataLoader", # IRIS Class 
                                   "SubmitResourceFiles", # Method name
                                   "/tmp/fhirdata", # Path to FHIR bundle (inside the IRIS container / server)
                                   "FHIRServer", # Type of load (Ignore)
                                   "/fhir/r4", # FHIR endpoint
                                   1, # Display progress
                                    "^FHIRLoadLog2" # Log progress to a global 
                                 )

print(status)
conn.close()
time_taken = time.time() - st
print(time_taken)
```
